In [8]:
# ============================================================
# RELATÓRIO DE PASSIVO — envio por E-MAIL + WHATSAPP (GARE11 e GAME11)
# ============================================================
import os
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from datetime import datetime
from dotenv import load_dotenv

# Helpers Z-API já usados pelo relatório ATIVO — importados, não duplicados.
from resumo_ativo import enviar_texto_whatsapp, enviar_documento_whatsapp

load_dotenv()

# Grupo "Guardian - PASSIVO inteligência listados" (formato: <id>-group).
GRUPO_PASSIVO_PHONE = os.getenv("GRUPO_PASSIVO_PHONE")

In [9]:
# ============================================================
# 1) DATA E CAMINHOS DOS PDFs — único lugar onde a data é editada
# ============================================================
data = "15.10.2026"   # ou: datetime.today().strftime("%d.%m.%Y")

DIARIOS = (
    r"g:\.shortcut-targets-by-id\1YZLLTFOuLniE6hO7WNHXl2c3MSa4TOv6\X. GAME11"
    r"\GUARDIAN Institucional\5. Planejamento & Processos\2. 2023\4. Research\2025"
    r"\5. Inteligência Passivos Guardian\2. RELATÓRIOS\1. DIARIOS"
)

# Atenção: o GARE exporta "Passivos_" (plural) e o GAME "Passivo_" (singular).
pdf_gare = rf"{DIARIOS}\GARE\Passivos_GARE11_{data}.pdf"
pdf_game = rf"{DIARIOS}\GAME\Passivo_GAME11_{data}.pdf"

In [10]:
# ============================================================
# 2) TEXTOS DO DIA ✍️ — a ÚNICA célula que muda todo dia
# ------------------------------------------------------------
#   CORPO_*  -> miolo HTML do e-mail (só as tags <p>; o invólucro
#               <html><body> e a assinatura entram em _html_email, célula 3).
#               Em HTML, use &gt; e &lt; no lugar de > e < (ex.: &gt;15M).
#   WHATS_*  -> texto do WhatsApp, enviado VERBATIM (emojis e "> " normais).
# ============================================================

# ---------------------------- GARE11 ----------------------------
CORPO_GARE = """
<p>Olá, time,</p>

<p>Resumo do cenário diário GARE11 — <b>15/10/2026</b>: cota em alta de <b>0,81%</b> contra o IFIX Tijolo estável, fechando a <b>R$ 8,76</b>, com o UBS comprando R$ 835k na faixa 3M-15M e a Patria seguindo na venda em &gt;15M. Relatório completo em anexo.</p>

<p><b>📊 Desempenho do Dia</b><br>
GARE11: 0,81% | IFIX Tijolo: 0,0% | Daytrade: 34% do volume total</p>

<p><b>📈 Compras</b> (TOP 10 – R$ 1,4 mi)<br>
5k-20k: 22,68% | 3M-15M: 21,30% | 1k-5k: 15,95%</p>

<p>O UBS AG, London Branch comprou R$ 835k (2º dia consecutivo) e se manteve na faixa 3M-15M, sozinho entre os institucionais dos 5 maiores compradores. Entraram 6 novos cotistas no dia, que somaram R$ 406k em compras — entre eles o JP Morgan Securities PLC, que voltou a comprar (R$ 52k, faixa 50k-100k) após zerar a posição em 08/10. Um cotista da faixa 5k-20k comprou R$ 35k e migrou para 20k-50k.</p>

<p><b>📉 Vendas</b> (TOP 10 – R$ 2,0 mi)<br>
&gt;15M: 28,54% | 20k-50k: 16,47% | 5k-20k: 14,63%</p>

<p>A Patria Renda Urbana (R$ 693k, 12º dia consecutivo) e a Patria Recebíveis Imobiliários (R$ 467k, 23º dia consecutivo) seguem vendendo na faixa &gt;15M e somaram R$ 1,2 mi. Um cotista da faixa 500k-1M vendeu R$ 238k e migrou para 100k-500k. Um cotista da faixa 100k-500k zerou novamente a posição, com venda de R$ 149k, e quatro cotistas da faixa 50k-100k também zeraram, somando R$ 267k.</p>
"""

WHATS_GARE = """📅 GARE11 - 15/10/2026

⚠️ XP E MORGAN STANLEY LIDERAM SALDOS NEGATIVOS EM DIA DE ALTA DA COTA CONTRA O IFIX TIJOLO, COM UBS COMPRANDO R$ 835K NA FAIXA 3M-15M E PATRIA SEGUINDO NA VENDA EM >15M

📈 Volume negociado foi de R$ 6,1 milhões. A cota fechou o dia em alta de 0,81%, a R$ 8,76. O fundo registrou mais um dia de entrada líquida de cotistas (+484), mantendo-se acima do patamar de 568 mil

📊 Desempenho do Dia:

* GARE11: 0,81%
* IFIX TIJOLO: 0,0%
* Daytrade: 34% do volume total

📈 Compras (TOP 10 – R$ 1,4 mi)

* 5k-20k: 22,68%
* 3M-15M: 21,30%
* 1k-5k: 15,95%

🔹UBS AG, LONDON BRANCH compra R$ 835k (2º dia consecutivo) mantendo-se na faixa 3M-15M.
🔹6 novos cotistas no dia, somando R$ 406k em compras.
🔹JP MORGAN SECURITIES PLC volta a comprar R$ 52k na faixa 50k-100k, após zerar a posição em 08/10.
🔹Cotista da faixa 5k-20k compra R$ 35k, migrando para 20k-50k.

📉 Vendas (TOP 10 – R$ 2,0 mi)

* >15M: 28,54%
* 20k-50k: 16,47%
* 5k-20k: 14,63%

🔹PATRIA RENDA URBANA (R$ 693k, 12º dia consecutivo) e PATRIA RECEBIVEIS IMOBILIARIOS (R$ 467k, 23º dia consecutivo) seguem vendendo na faixa >15M, somando R$ 1,2 mi.
🔹Cotista da faixa 500k-1M vende R$ 238k, migrando para 100k-500k.
🔹Cotista da faixa 100k-500k zera novamente a posição com venda de R$ 149k.
🔹Quatro cotistas da faixa 50k-100k zeram a posição, somando R$ 267k."""


# ---------------------------- GAME11 ----------------------------
# Mesma estrutura do GARE11 — preencher com os números do dia.
CORPO_GAME = """
<p>Olá, time,</p>

<p>Resumo do cenário diário GAME11 — <b>15/10/2026</b>: cota em alta de <b>2,22%</b> acompanhando o IFIX Papel (0,3%), fechando a <b>R$ 8,76</b>, com a Inter Hedge comprando R$ 88k na faixa 100k-500k e o FII RBR Rendimento High Grade vendendo R$ 126k em 3M-15M. Relatório completo em anexo.</p>

<p><b>📊 Desempenho do Dia</b><br>
GAME11: 2,22% | IFIX Papel: 0,3% | Daytrade: 22% do volume total</p>

<p><b>📈 Compras</b> (TOP 10 – R$ 181k)<br>
100k-500k: 43,46% | &gt;15M: 39,11% | 0-1k: 5,91%</p>

<p>Três institucionais ficaram entre os 5 maiores compradores: o Inter Hedge Fundo de Investimento Imobiliário comprou R$ 88k (5º dia consecutivo) e se manteve na faixa 100k-500k; o Guardian Strategy FII de RL comprou R$ 79k (2º dia consecutivo) e se manteve em &gt;15M; e a XP Investimentos CCTVM, que vendeu ontem, comprou R$ 4k e se manteve em 5k-20k. Nenhum novo cotista entre os 10 maiores compradores. Um cotista da faixa 0-1k comprou R$ 3k e migrou para 1k-5k.</p>

<p><b>📉 Vendas</b> (TOP 10 – R$ 184k)<br>
3M-15M: 62,28% | 5k-20k: 11,92% | 100k-500k: 10,86%</p>

<p>O FII RBR Rendimento High Grade vendeu R$ 126k (2º dia consecutivo) e se manteve na faixa 3M-15M, único institucional entre os 5 maiores vendedores. Um cotista da faixa 100k-500k vendeu R$ 22k (3º dia consecutivo) e se manteve na faixa. Cinco cotistas das faixas 5k-20k e 1k-5k zeraram a posição, somando R$ 21k.</p>
"""

WHATS_GAME = """📅 GAME11 - 15/10/2026

⚠️ XP E BTG LIDERAM SALDOS NEGATIVOS EM DIA DE ALTA DA COTA ACOMPANHANDO O IFIX PAPEL, COM INTER HEDGE COMPRANDO R$ 88K NA FAIXA 100K-500K E FII RBR RENDIMENTO HIGH GRADE VENDENDO R$ 126K EM 3M-15M

📈 Volume negociado foi de R$ 250 mil. A cota fechou o dia em alta de 2,22%, a R$ 8,76. O fundo registrou saída líquida de cotistas (-8), mantendo-se acima do patamar de 20 mil

📊 Desempenho do Dia:

* GAME11: 2,22%
* IFIX PAPEL: 0,3%
* Daytrade: 22% do volume total

📈 Compras (TOP 10 – R$ 181k)

* 100k-500k: 43,46%
* >15M: 39,11%
* 0-1k: 5,91%

🔹INTER HEDGE FUNDO DE INVESTIMENTO IMOBILIÁRIO compra R$ 88k (5º dia consecutivo) mantendo-se na faixa 100k-500k.
🔹GUARDIAN STRATEGY FII DE RL compra R$ 79k (2º dia consecutivo) mantendo-se na faixa >15M.
🔹XP INVESTIMENTOS CCTVM, que vendeu ontem, compra R$ 4k mantendo-se na faixa 5k-20k.
🔹Nenhum novo cotista entre os 10 maiores compradores.

📉 Vendas (TOP 10 – R$ 184k)

* 3M-15M: 62,28%
* 5k-20k: 11,92%
* 100k-500k: 10,86%

🔹FII RBR RENDIMENTO HIGH GRADE vende R$ 126k (2º dia consecutivo) mantendo-se na faixa 3M-15M.
🔹Cotista da faixa 100k-500k vende R$ 22k (3º dia consecutivo) mantendo-se na faixa.
🔹Cinco cotistas das faixas 5k-20k e 1k-5k zeram a posição, somando R$ 21k."""

In [11]:
# ============================================================
# 3) FUNÇÕES DE ENVIO — uma implementação, usada pelos dois fundos
# ============================================================
REMETENTE = "lc@guardian-asset.com"

DESTINATARIOS = [
    "lc@guardian-asset.com", "pmk@guardian-asset.com", "msf@guardian-asset.com",
    "cm@guardian-asset.com", "lb@guardian-asset.com", "ja@guardian-asset.com",
    "gsa@guardian-asset.com", "gvm@guardian-asset.com", "rt@guardian-asset.com",
    "acl@guardian-asset.com", "lt@guardian-asset.com", "tfa@guardian-asset.com", "ffs@guardian-asset.com",
]

# Fecha todo e-mail, nos dois fundos (anexada por _html_email).
ASSINATURA = """<p>Att,<br>

Lucas Cavalcante.</p>"""


def _html_email(corpo):
    """Embrulha o miolo HTML no mesmo estilo usado no relatório ATIVO."""
    return (
        '<html><body style="font-family: Arial, sans-serif; font-size: 14px; '
        'color: #222222; line-height: 1.55;">'
        f"{corpo.strip()}"
        f"{ASSINATURA}"
        "</body></html>"
    )


def _enviar_email(pdf_path, corpo, assunto):
    msg = MIMEMultipart()
    msg["From"] = REMETENTE
    msg["To"] = ", ".join(DESTINATARIOS)
    msg["Subject"] = assunto
    msg.attach(MIMEText(_html_email(corpo), "html"))

    # Anexar o PDF
    try:
        with open(pdf_path, "rb") as attachment:
            part = MIMEBase("application", "octet-stream")
            part.set_payload(attachment.read())
        encoders.encode_base64(part)
        part.add_header(
            "Content-Disposition",
            f"attachment; filename= {os.path.basename(pdf_path)}",
        )
        msg.attach(part)
    except Exception as e:
        print(f"❌ Falha ao anexar o arquivo: {e}")
        return

    # Enviar via SMTP do Gmail
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp_server:
            smtp_server.login(REMETENTE, os.getenv("senha_gmail"))
            smtp_server.sendmail(REMETENTE, DESTINATARIOS, msg.as_string())
        print(f"✅ E-mail enviado — {assunto}")
    except Exception as e:
        print(f"❌ Falha ao enviar e-mail: {e}")


def _enviar_whatsapp(texto, pdf_path):
    """Envia o texto e, em seguida, o PDF ao grupo do PASSIVO."""
    if not GRUPO_PASSIVO_PHONE:
        raise ValueError("GRUPO_PASSIVO_PHONE não definido no .env.")
    enviar_texto_whatsapp(texto, phone=GRUPO_PASSIVO_PHONE)
    enviar_documento_whatsapp(pdf_path, phone=GRUPO_PASSIVO_PHONE)


def enviar_passivo(fundo, pdf_path, corpo, texto_whats, data, email=True, whatsapp=True):
    """fundo: 'GARE' ou 'GAME'. Assunto: RELATÓRIO DE PASSIVO {fundo} DIA {data}."""
    if email:
        _enviar_email(pdf_path, corpo, f"RELATÓRIO DE PASSIVO {fundo} DIA {data}")
    if whatsapp:
        _enviar_whatsapp(texto_whats, pdf_path)

In [12]:
# ============================================================
# 4) 👀 PRÉ-VISUALIZAÇÃO — não envia nada. Rode antes das células 5 e 6.
# ============================================================
from IPython.display import HTML, display

for fundo, pdf_path, corpo, texto in [
    ("GARE", pdf_gare, CORPO_GARE, WHATS_GARE),
    ("GAME", pdf_game, CORPO_GAME, WHATS_GAME),
]:
    print("=" * 78)
    print(f"Assunto : RELATÓRIO DE PASSIVO {fundo} DIA {data}")
    print(f"Anexo   : {'✅' if os.path.exists(pdf_path) else '❌ NÃO ENCONTRADO'}  {pdf_path}")
    print(f"Grupo   : {GRUPO_PASSIVO_PHONE}")
    print("-" * 78)
    display(HTML(_html_email(corpo)))
    print("--- WhatsApp ".ljust(78, "-"))
    print(texto)
    print()

Assunto : RELATÓRIO DE PASSIVO GARE DIA 15.10.2026
Anexo   : ✅  g:\.shortcut-targets-by-id\1YZLLTFOuLniE6hO7WNHXl2c3MSa4TOv6\X. GAME11\GUARDIAN Institucional\5. Planejamento & Processos\2. 2023\4. Research\2025\5. Inteligência Passivos Guardian\2. RELATÓRIOS\1. DIARIOS\GARE\Passivos_GARE11_15.10.2026.pdf
Grupo   : 120363370142403605-group
------------------------------------------------------------------------------


--- WhatsApp -----------------------------------------------------------------
📅 GARE11 - 15/10/2026

⚠️ XP E MORGAN STANLEY LIDERAM SALDOS NEGATIVOS EM DIA DE ALTA DA COTA CONTRA O IFIX TIJOLO, COM UBS COMPRANDO R$ 835K NA FAIXA 3M-15M E PATRIA SEGUINDO NA VENDA EM >15M

📈 Volume negociado foi de R$ 6,1 milhões. A cota fechou o dia em alta de 0,81%, a R$ 8,76. O fundo registrou mais um dia de entrada líquida de cotistas (+484), mantendo-se acima do patamar de 568 mil

📊 Desempenho do Dia:

* GARE11: 0,81%
* IFIX TIJOLO: 0,0%
* Daytrade: 34% do volume total

📈 Compras (TOP 10 – R$ 1,4 mi)

* 5k-20k: 22,68%
* 3M-15M: 21,30%
* 1k-5k: 15,95%

🔹UBS AG, LONDON BRANCH compra R$ 835k (2º dia consecutivo) mantendo-se na faixa 3M-15M.
🔹6 novos cotistas no dia, somando R$ 406k em compras.
🔹JP MORGAN SECURITIES PLC volta a comprar R$ 52k na faixa 50k-100k, após zerar a posição em 08/10.
🔹Cotista da faixa 5k-20k compra R$ 35k, migrando para 20k-50k.

📉 Vendas (TOP 10 – R$ 2,0 mi)

* >15M: 28,54%
*

--- WhatsApp -----------------------------------------------------------------
📅 GAME11 - 15/10/2026

⚠️ XP E BTG LIDERAM SALDOS NEGATIVOS EM DIA DE ALTA DA COTA ACOMPANHANDO O IFIX PAPEL, COM INTER HEDGE COMPRANDO R$ 88K NA FAIXA 100K-500K E FII RBR RENDIMENTO HIGH GRADE VENDENDO R$ 126K EM 3M-15M

📈 Volume negociado foi de R$ 250 mil. A cota fechou o dia em alta de 2,22%, a R$ 8,76. O fundo registrou saída líquida de cotistas (-8), mantendo-se acima do patamar de 20 mil

📊 Desempenho do Dia:

* GAME11: 2,22%
* IFIX PAPEL: 0,3%
* Daytrade: 22% do volume total

📈 Compras (TOP 10 – R$ 181k)

* 100k-500k: 43,46%
* >15M: 39,11%
* 0-1k: 5,91%

🔹INTER HEDGE FUNDO DE INVESTIMENTO IMOBILIÁRIO compra R$ 88k (5º dia consecutivo) mantendo-se na faixa 100k-500k.
🔹GUARDIAN STRATEGY FII DE RL compra R$ 79k (2º dia consecutivo) mantendo-se na faixa >15M.
🔹XP INVESTIMENTOS CCTVM, que vendeu ontem, compra R$ 4k mantendo-se na faixa 5k-20k.
🔹Nenhum novo cotista entre os 10 maiores compradores.

📉 Venda

In [13]:
# ============================================================
# 5) ENVIAR E-MAILS — confira a célula 4 antes de rodar
# ============================================================
enviar_passivo("GARE", pdf_gare, CORPO_GARE, WHATS_GARE, data, whatsapp=False)
enviar_passivo("GAME", pdf_game, CORPO_GAME, WHATS_GAME, data, whatsapp=False)

✅ E-mail enviado — RELATÓRIO DE PASSIVO GARE DIA 15.10.2026
✅ E-mail enviado — RELATÓRIO DE PASSIVO GAME DIA 15.10.2026


In [14]:
# ============================================================
# 6) ENVIAR AO WHATSAPP — grupo "Guardian - PASSIVO inteligência listados"
#    4 mensagens: texto GARE -> PDF GARE -> texto GAME -> PDF GAME
# ============================================================
enviar_passivo("GARE", pdf_gare, CORPO_GARE, WHATS_GARE, data, email=False)
enviar_passivo("GAME", pdf_game, CORPO_GAME, WHATS_GAME, data, email=False)

✅ Texto enviado ao WhatsApp (120363370142403605-group). messageId: 3EB0E97EE94AEF9406DE6D
✅ PDF enviado ao WhatsApp (120363370142403605-group). messageId: 3EB00C4CBAE68705EE57F6
✅ Texto enviado ao WhatsApp (120363370142403605-group). messageId: 3EB0F0A45F1A022D93941A
✅ PDF enviado ao WhatsApp (120363370142403605-group). messageId: 3EB0B8BE5EABC093F68CC0
